# Validation of GRS with AERONET-OC on the Sentinel-2 subsets

Matchups of the GRS L2A remote sensing reflectance with the AERONET-OC measurements, on the Sentinel-2 L1C
subsets of 5 km x 5 km around the AERONET-OC stations (bucket `s3://aeronet-oc`, content in
`/data/satellite/validation`, see `aeronet_oc_s2_l1c_subsets.ipynb`), with {mod}`grstbx.validation`:

1. **subsets** of the STAC catalogue ({func}`~grstbx.validation.read_items`), with the AERONET-OC measurements
   around the overpass;
2. **GRS** on the subsets without L2A product ({func}`~grstbx.validation.process_subsets`, one process per image,
   L2A products with their STAC Item in `sentinel-2/l2a/<station>/YYYY/MM/DD/<name>/` of the bucket);
3. **matchup table** ({func}`~grstbx.validation.matchup_table`): one row per image and Sentinel-2 band;
   - satellite: median and standard deviation of the valid water pixels (`mask == 0`) of a **ring 4 to 7 pixels**
     (80 to 140 m) around the station, without the platform and its shadow (pixels brighter than the median at
     865 nm);
   - in situ: the AERONET-OC measurement nearest in time (the nearest **Level 2.0** one if any, else Level 1.5) of
     the validation database, at the nearest AERONET-OC band within 15 nm; the three normalisations: `Rrs_insitu`
     (Lwn / F0, directional as measured: closest to the directional Rrs of GRS), `Rrs_insitu_fQ` (Morel et al.
     2002), `Rrs_insitu_IOP` (Lee et al. 2011);
4. **quality criteria** ({func}`~grstbx.validation.quality`), kept as columns of the table so that they can be
   changed afterwards: lag of **1 h** at most (Zibordi et al. 2009), at least **50 %** of the window valid,
   **CV(Rrs 560) < 0.2** in the window (Bailey and Werdell 2006), Level 2.0;
5. **statistics** ({func}`~grstbx.validation.metrics_table`, {func}`grstbx.matchup.matchup_metrics`): regression,
   RMSE, bias, MAE, MdSA and SSPB (Morley et al. 2018), MedAPE and log-space RMSE / bias / MAE (as satmatchup and
   ACIX);
6. **dashboard** ({func}`~grstbx.validation.dashboard`): the criteria and selections changed interactively, scatter
   plots, statistics, errors, spectra and quicklooks, time series and drivers of the residuals.

The same chain from the command line:

```bash
grstbx-validation grs --stations AAOT --workers 2          # GRS on the subsets without L2A product
grstbx-validation matchups -o /data/satellite/validation/matchups/grs_3.1.1.csv
grstbx-validation dashboard /data/satellite/validation/matchups/grs_3.1.1.csv --show
```

In [ ]:
# development versions of the GRS suite (repositories), before the installed packages, also for the
# processes started by this notebook; remove to use the installed versions (GRS >= 3.1.1, GRSdriver >= 2.1.1)
import os, sys
DEV_REPOS = [f'/home/harmel/Dropbox/Dropbox/work/git/satellite_app/{repo}'
             for repo in ('grs', 'GRSl2bgen', 'GRSdriver', 'grstbx')]
DEV_REPOS = [repo for repo in DEV_REPOS if os.path.isdir(repo)]
sys.path[:0] = DEV_REPOS
os.environ['PYTHONPATH'] = os.pathsep.join(DEV_REPOS + [os.environ.get('PYTHONPATH', '')]).strip(os.pathsep)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import grs
from grstbx import validation as val
from grstbx.matchup import plot_matchup

print('GRS', grs.__version__)

## Settings

In [ ]:
STATIONS = ['AAOT']           # None: all the stations of the catalogue
START, END = None, None       # period of the images (None: all)
MAX_DT_MIN = 60               # GRS run only on the images with an AERONET-OC measurement within this lag (min)
RUN_GRS = True                # False: matchups of the L2A products already there only
WORKERS = 2                   # GRS processes at the same time (about 1 to 2 GB each)
GRS_OPTIONS = {}              # options of grs.Process.execute, e.g. {'glint_aot': True}
OUT = Path(val.DATA_ROOT) / 'matchups' / f'grs_{grs.__version__}.csv'

## 1. Subsets of the catalogue

In [ ]:
items = val.read_items(stations=STATIONS, start=START, end=END)
items = items[items.dt_min.abs() <= MAX_DT_MIN]
print(f'{len(items)} subsets with an AERONET-OC measurement within {MAX_DT_MIN} min')
items.groupby('station').agg(images=('id', 'size'), with_lev20=('n_obs_lev20', lambda n: int((n > 0).sum())),
                             first=('time', 'min'), last=('time', 'max'))

## 2. GRS on the subsets

One process per image (`grstbx-validation grs-one`), `WORKERS` at a time; the L2A products already there are
reused, the images without CAMS file are skipped, a failure leaves a `.err` file next to the product (deleted, or
`retry_failed=True`, to retry), an image without water (cloudy) a `_no_water.txt` marker.

In [ ]:
if RUN_GRS:
    runs = val.process_subsets(items, workers=WORKERS, options=GRS_OPTIONS)
    print(runs.state.value_counts().to_dict())

## 3. Matchup table

In [ ]:
table = val.matchup_table(items)
val.save_table(table, OUT)
ok = val.quality(table)
print(f'{table.image.nunique()} L2A products, {len(table)} rows (image x band), {ok.sum()} meeting the criteria '
      f'({table[ok].image.nunique()} images) -> {OUT}')
table[ok].head()

## 4. Statistics (default criteria)

In [ ]:
val.metrics_table(table[ok], by='wl').round(5)

In [ ]:
val.metrics_table(table[ok], by='station').round(5)

In [ ]:
sel = table[ok]
bands = sorted(sel.wl.unique())
fig, axs = plt.subplots(2, (len(bands) + 1) // 2, figsize=(4.2 * ((len(bands) + 1) // 2), 8.4), squeeze=False)
for ax, wl in zip(axs.flat, bands):
    g = sel[sel.wl == wl]
    vmax = 1.1 * max(g.Rrs_insitu.max(), g.Rrs_sat.max())
    plot_matchup(ax, g.Rrs_insitu, g.Rrs_sat, min(0, g.Rrs_sat.min()), vmax, yerr=g.Rrs_sat_std, s=18, c='k', alpha=0.6)
    ax.set(title=f'{wl} nm (AERONET-OC {int(g.wl_insitu.iloc[0])} nm)', xlabel='Rrs AERONET-OC (sr$^{-1}$)',
           ylabel='Rrs GRS (sr$^{-1}$)')
for ax in list(axs.flat)[len(bands):]:
    ax.set_visible(False)
fig.suptitle(f'GRS {grs.__version__} vs AERONET-OC (Level 2.0, lag <= 1 h, ring 80-140 m)')
fig.tight_layout()

## 5. Dashboard

Interactive analysis of the table: selection of the stations, bands, in situ level and normalisation, period,
platforms and GRS versions, quality criteria (lag, valid fraction, CV, AOT, sunglint); tabs Overview (map and
statistics), Scatter, Statistics (tables and heat map), Errors, Spectra (with the quicklooks of the L2A
products), Time series and Drivers; download of the selected matchups. Served by the kernel of this notebook;
as a web application: `grstbx-validation dashboard <table.csv> --show` (several tables, e.g. two GRS versions,
can be compared).

In [ ]:
val.dashboard(table)